In [ ]:
import scanpy as sc

In [ ]:
adata = sc.read_h5ad("c:/Users/irc/Desktop/Internship Bioinformatics 2025-2026/Lode/Integration_scVI/scVI_integrated_object.h5ad")

In [ ]:
adata.layers["raw_counts"] = adata.X

In [ ]:
adata

Normalization, scaling etc

In [ ]:
sc.pp.highly_variable_genes(adata,
                            n_top_genes=4000,
                            flavor = "seurat_v3")
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

For visualization

In [ ]:
# use latent space of scVI
sc.pp.pca(adata, n_comps=50, use_highly_variable=True, svd_solver="arpack")
sc.pp.neighbors(adata, use_rep="X_scVI", n_neighbors=15, n_pcs=30)
sc.tl.umap(adata)
sc.tl.leiden(adata, 
             key_added="leiden_1", 
             n_iterations=2, 
             directed=False)

In [ ]:
sc.pl.umap(adata,
           color=["orig.ident", "celltype_new", "experiment", "leiden_1", "treatment"],
           frameon=False,
           ncols=2)

Now, the annotation based on marker genes and DE

In [ ]:
markers = {
    "Pre_cDC1": ["Ccr2", "Fcer1g", "Cd24a", "Vim"],
    "Early_Immature": ["Sell", "Creld2", "Pdia4"],
    "Late_Immature": ["Cd207", "Itgae", "Apol7c", "Apoe", "Dnase1l3", "Cadm1", "Xcr1", "Cd83", "Cd86"],
    "Early_Mature": ["Cxcl10", "Cxcl9", "Iigp1", "Ifi47", "Gbp2", "Gbp5", "Cd40"],
    "Late_Mature": ["Cd63", "Fscn1", "Il4i1", "Socs2", "Ccr7"]
}

In [ ]:
marker_genes_in_data = {}
for ct, markers in markers.items():
    markers_found = []
    for marker in markers:
        if marker in adata.var.index:
            markers_found.append(marker)
    marker_genes_in_data[ct] = markers_found

Dotplot

In [ ]:
sc.pl.dotplot(
    adata,
    groupby="leiden_1",
    var_names="markers",
    standard_scale="var",  # standard scale: normalize each gene to range from 0 to 1
)

Now perform Differential expression for annotation

In [ ]:
sc.tl.rank_genes_groups(
    adata, groupby="leiden_1", method="wilcoxon", key_added="dea_leiden_1"
)

In [ ]:
sc.tl.dendrogram(
    adata,
    groupby="leiden_1",
)

sc.pl.rank_genes_groups_dotplot(
    adata, groupby="leiden_1", standard_scale="var", n_genes=5, key="dea_leiden_1"
)

Filtered DE

In [ ]:
sc.tl.filter_rank_genes_groups(
    adata,
    min_in_group_fraction=0.2,
    max_out_group_fraction=0.2,
    key="dea_leiden_1",
    key_added="dea_leiden_1_filtered",
)

In [ ]:
sc.pl.rank_genes_groups_dotplot(
    adata,
    groupby="leiden_1",
    standard_scale="var",
    n_genes=5,
    key="dea_leiden_1_filtered",
)